In [3]:
"""
Pobiera z OpenStreetMap (przez Overpass API) nowe typy kandydackich
lokalizacji pod stacje ladowania w segmencie docelowym - centra handlowe,
duze supermarkety, duze parkingi ogolne.

UWAGA: overpass-api.de jest zablokowany w moim srodowisku (403) - ten
skrypt trzeba uruchomic lokalnie, u siebie.

Wymaga: requests (pip install requests)
Wynik: nowe_typy_osm.csv
"""

import os
import time
import requests
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Kazdy typ - osobne zapytanie Overpass QL, ograniczone do obszaru Polski
# (area["ISO3166-1"="PL"] - standardowy sposob ograniczenia do calego kraju)
ZAPYTANIA = {
    "centrum_handlowe": '''
        [out:json][timeout:180];
        area["ISO3166-1"="PL"]->.polska;
        way["shop"="mall"](area.polska);
        out geom;
    ''',
    # UWAGA: pobieramy TYLKO 'way' (nie 'node') - filtr jakosci i tak
    # odrzuca node'y (nie maja geometrii budynku do zmierzenia), wiec ich
    # pobieranie bylo zmarnowana praca, obciazajaca ciezkie zapytanie
    # "out geom" bez zadnej korzysci. To byla najprawdopodobniejsza
    # przyczyna powtarzajacych sie bledow 504 (przeciazenie serwera) -
    # zapytanie bylo cieszsze niz musialo.
    "supermarket": '''
        [out:json][timeout:180];
        area["ISO3166-1"="PL"]->.polska;
        (
          node["shop"="supermarket"](area.polska);
          way["shop"="supermarket"](area.polska);
        );
        out center;
    ''',
    # UWAGA: filtr zluzowany do WSZYSTKICH supermarketow (usunieto
    # ["building:levels"] i ["parking"="surface"] - te tagi okazaly sie
    # rzadko uzywane w polskich danych OSM, dawaly tylko 7 wynikow zamiast
    # spodziewanych setek/tysiecy). Brak w OSM wiarygodnego, powszechnie
    # stosowanego tagu "wielkosci" supermarketu - to zlapie WSZYSTKIE, w
    # tym mniejsze, ale to lepsze niz prawie pusty wynik. Rozmiar mozna
    # pozniej przyblizyc np. przez pole powierzchni budynku (way), jesli
    # okaze sie to potrzebne.
    "parking_duzy": '''
        [out:json][timeout:180];
        area["ISO3166-1"="PL"]->.polska;
        (
          node["amenity"="parking"]["capacity"](area.polska);
          way["amenity"="parking"]["capacity"](area.polska);
        );
        out center;
    ''',
}


def powierzchnia_m2(geometria):
    """Powierzchnia wielokata (lista {lat,lon}) w przyblizeniu plaskim -
    wystarczajaco dokladne dla obiektow wielkosci pojedynczego budynku
    (rzedu setek metrow), gdzie krzywizna Ziemi jest pomijalna."""
    if not geometria or len(geometria) < 3:
        return None
    lat_srednia = sum(p["lat"] for p in geometria) / len(geometria)
    m_na_stopien_lat = 111320
    m_na_stopien_lon = 111320 * np.cos(np.radians(lat_srednia))
    x = [p["lon"] * m_na_stopien_lon for p in geometria]
    y = [p["lat"] * m_na_stopien_lat for p in geometria]
    n = len(x)
    suma = sum(x[i] * y[(i + 1) % n] - x[(i + 1) % n] * y[i] for i in range(n))
    return abs(suma) / 2


def pobierz_typ(nazwa_typu, zapytanie):
    print(f"Pobieram: {nazwa_typu}...")
    naglowki = {
        "Content-Type": "application/x-www-form-urlencoded",
        "User-Agent": "projekt-ev-charging-pl/1.0 (skrypt badawczy, kontakt: [twoj e-mail])",
    }
    odpowiedz = requests.post(OVERPASS_URL, data={"data": zapytanie}, headers=naglowki, timeout=200)

    if odpowiedz.status_code != 200:
        # DIAGNOSTYKA: Overpass zazwyczaj dokladnie tlumaczy w TRESCI
        # odpowiedzi, co jest nie tak (np. konkretna linijka bledu skladni).
        # raise_for_status() by to ukryl, wiec wypisujemy to jawnie.
        print(f"  STATUS: {odpowiedz.status_code}")
        print(f"  TRESC ODPOWIEDZI (pierwsze 2000 znakow):")
        print(f"  {odpowiedz.text[:2000]}")
        odpowiedz.raise_for_status()

    dane = odpowiedz.json()["elements"]

    wyniki = []
    for element in dane:
        # 'node' ma lat/lon bezposrednio; 'way' z 'out geom' ma liste
        # wspolrzednych w 'geometry' (uzywana do liczenia powierzchni),
        # z 'out center' mialby zamiast tego pole 'center' (starsze
        # zapytania, ktore tego nie potrzebuja, nadal go uzywaja)
        geometria = None
        if element["type"] == "node":
            lat, lon = element.get("lat"), element.get("lon")
        elif "geometry" in element:
            punkty = element["geometry"]
            if punkty:
                lat = sum(p["lat"] for p in punkty) / len(punkty)
                lon = sum(p["lon"] for p in punkty) / len(punkty)
                geometria = punkty
            else:
                lat = lon = None
        else:
            centrum = element.get("center", {})
            lat, lon = centrum.get("lat"), centrum.get("lon")
        if lat is None or lon is None:
            continue

        tagi = element.get("tags", {})
        wyniki.append({
            "osm_id": element["id"],
            "osm_type": element["type"],
            "source_layer": nazwa_typu,
            "name": tagi.get("name"),
            "capacity": tagi.get("capacity"),
            "access": tagi.get("access"),
            "brand": tagi.get("brand"),
            "stars": tagi.get("stars"),
            "rooms": tagi.get("rooms"),
            # liczymy powierzchnie OD RAZU (nie trzymamy calej geometrii -
            # lista punktow nie przetrwalaby zapisu/odczytu z CSV jako
            # prawdziwa struktura danych, tylko jako bezuzyteczny tekst)
            "powierzchnia_m2": powierzchnia_m2(geometria),
            "latitude": lat,
            "longitude": lon,
        })
    print(f"  Znaleziono: {len(wyniki)}")
    return wyniki


# Kody bledow, ktore oznaczaja "sprobuj pozniej", nie "cos jest zle w zapytaniu"
KODY_DO_PONOWIENIA = {429, 502, 503, 504}


def pobierz_z_ponawianiem(nazwa_typu, zapytanie, maks_prob=4):
    """Ponawia zapytanie przy przejsciowych bledach serwera (429 rate limit,
    504 przeciazenie itp.), czekajac coraz dluzej (30s, 60s, 90s, 120s)."""
    for proba in range(1, maks_prob + 1):
        try:
            return pobierz_typ(nazwa_typu, zapytanie)
        except requests.exceptions.HTTPError as e:
            kod_bledu = e.response.status_code if e.response is not None else None
            if kod_bledu in KODY_DO_PONOWIENIA and proba < maks_prob:
                czas_oczekiwania = 30 * proba
                print(f"  Blad {kod_bledu} (przejsciowy) - czekam {czas_oczekiwania}s przed proba {proba+1}/{maks_prob}...")
                time.sleep(czas_oczekiwania)
            else:
                raise
    return []


PLIK_SUROWY = "../data/nowe_typy_osm_surowe.csv" if os.path.exists("../data") else "nowe_typy_osm_surowe.csv"
PLIK_WYNIKOWY = "../data/nowe_typy_osm.csv" if os.path.exists("../data") else "nowe_typy_osm.csv"
# PLIK_SUROWY: NIGDY nie czyszczony - tylko gromadzony (dane z Overpass)
# PLIK_WYNIKOWY: ZAWSZE przeliczany od nowa z PLIK_SUROWY (efekt oczysc_dane())

# Progi czyszczenia danych - patrz funkcje oczysc_dane() nizej
PROG_POJEMNOSCI_PARKING = 50   # min. liczba miejsc, zeby uznac parking za "duzy"
PROG_POWIERZCHNI_CENTRUM_M2 = 5000  # min. powierzchnia budynku, zeby uznac za "duze" centrum handlowe

# Znane, duze sieci supermarketow/hipermarketow w Polsce - jedyny wiarygodny
# sygnal wielkosci, skoro OSM nie ma tagu "duzy supermarket". Dopasowanie
# czesciowe, bez wielkosci liter (np. zlapie "Kaufland Wroclaw Bielany").
DUZE_SIECI_SUPERMARKETOW = [
    "biedronka", "lidl", "kaufland", "carrefour", "auchan", "netto",
    "dino", "e.leclerc", "leclerc", "intermarche", "intermarché",
    "tesco", "stokrotka",
]


def przejdz_filtr_jakosci(wiersz):
    """Filtr 'czy to w ogole realny kandydat' - osobne kryterium per typ,
    bo kazdy tag OSM ma inny, najbardziej wiarygodny sygnal wielkosci."""
    typ = wiersz["source_layer"]

    if typ == "centrum_handlowe":
        # tylko 'way' z policzalna powierzchnia - punkty ('node') bez
        # zarysu budynku nie daja sposobu na weryfikacje wielkosci
        pow = wiersz.get("powierzchnia_m2")
        return pd.notna(pow) and pow >= PROG_POWIERZCHNI_CENTRUM_M2

    if typ == "supermarket":
        marka = str(wiersz.get("brand", "")).lower()
        return any(duza_siec in marka for duza_siec in DUZE_SIECI_SUPERMARKETOW)

    return True  # parking_duzy ma juz wlasny, osobny filtr pojemnosci nizej
# Rozny promien per typ - duze kompleksy (galerie) czesto maja otagowane
# rozne wejscia/skrzydla ODLEGLE od siebie o wiecej niz typowe 150m, wiec
# dla nich uzywamy szerszego progu. Dla mniejszych obiektow (supermarket,
# parking) 150m wystarcza.
PROGI_DUPLIKATU_M = {
    "centrum_handlowe": 300,
    "parking_duzy": 150,
    "supermarket": 150,
}
PROG_DUPLIKATU_DOMYSLNY_M = 150


def znormalizuj_nazwe(nazwa):
    """Male litery, usuniete typowe dopiski (wejscie/parking/wjazd/skrzydlo),
    przyciete biale znaki - zeby "Galeria Posada" i "Galeria Posada -
    wejscie polnocne" dawaly ten sam, porownywalny rdzen nazwy."""
    if pd.isna(nazwa):
        return None
    n = str(nazwa).lower().strip()
    for dopisek in [" - wejscie", " wejscie", " - parking", " parking",
                    " - wjazd", " wjazd", " - skrzydlo", " skrzydlo",
                    " north", " south", " east", " west", " polnocne",
                    " poludniowe", " wschodnie", " zachodnie"]:
        n = n.split(dopisek)[0].strip()
    return n if n else None


def to_samo_miejsce(nazwa_a, nazwa_b):
    """Dwa punkty uznajemy za TEN SAM obiekt tylko jesli: (a) ktoras
    nazwa jest nieznana (nie da sie odroznic - lepiej ostroznie zalozyc
    duplikat), LUB (b) nazwy po normalizacji sa identyczne, LUB (c) jedna
    jest podciagiem drugiej. Dwie WYRAZNIE ROZNE nazwy w pobliskiej
    odleglosci NIE sa laczone - to prawdopodobnie naprawde dwa sasiadujace,
    osobne miejsca (np. dwa rozne sklepy przy tej samej ulicy)."""
    a, b = znormalizuj_nazwe(nazwa_a), znormalizuj_nazwe(nazwa_b)
    if a is None or b is None:
        return True
    if a == b:
        return True
    return a in b or b in a


def deduplikuj_typ(grupa, prog_m):
    """Usuwa punkty tego samego typu lezace blisko siebie (<prog_m) I
    majace pasujaca nazwe (patrz to_samo_miejsce) - prawdopodobnie ten
    sam fizyczny obiekt otagowany kilka razy w OSM. Blisko lezace punkty
    o WYRAZNIE roznych nazwach NIE sa laczone. Przy laczeniu duplikatow
    preferujemy zachowanie 'way' (prawdziwy zarys budynku) nad 'node'
    (pojedynczy punkt, np. wejscie)."""
    if len(grupa) <= 1:
        return grupa

    grupa = grupa.reset_index(drop=True)
    wspolrzedne_rad = np.radians(grupa[["latitude", "longitude"]].values)
    drzewo = BallTree(wspolrzedne_rad, metric="haversine")
    prog_rad = prog_m / 6371000  # promien Ziemi w metrach

    # kolejnosc przetwarzania: najpierw 'way' (preferowane jako "kotwica"
    # skupiska), potem 'node' - dzieki temu jesli way i node sa duplikatem,
    # to WAY przetrwa (jest przetwarzany pierwszy, node zostanie odrzucony
    # jako jego duplikat), nie odwrotnie
    kolejnosc = grupa.sort_values("osm_type", ascending=False).index.tolist()  # 'way' > 'node' alfabetycznie odwrotnie

    do_zachowania = np.ones(len(grupa), dtype=bool)
    for i in kolejnosc:
        if not do_zachowania[i]:
            continue
        sasiedzi = drzewo.query_radius(wspolrzedne_rad[i:i + 1], r=prog_rad)[0]
        for j in sasiedzi:
            if j == i or not do_zachowania[j]:
                continue
            if to_samo_miejsce(grupa.at[i, "name"], grupa.at[j, "name"]):
                do_zachowania[j] = False
    return grupa[do_zachowania]


def oczysc_dane(df):
    """Trzy etapy czyszczenia, uruchamiane na PELNYM (istniejace+nowe) zbiorze:
    1. Filtr jakosci per typ - patrz przejdz_filtr_jakosci(): powierzchnia
       budynku (centra handlowe), znana duza siec (supermarkety), formalna
       znana duza siec (supermarkety). Odsiewa realnie male obiekty otagowane tym
       samym, szerokim tagiem OSM co duze (np. piekarnia otagowana jako
       "supermarket").
    2. Parkingi ponizej realnego progu pojemnosci - odrzucone.
    3. Deduplikacja przestrzenna w obrebie kazdego typu z osobna - patrz
       deduplikuj_typ()."""
    print(f"\nCzyszczenie danych - przed: {len(df)} lokalizacji")

    df = df.copy()

    # ---------- etap 1: filtr jakosci/wielkosci per typ ----------
    maska_ma_wlasny_filtr = df["source_layer"].isin(["centrum_handlowe", "supermarket", "parking_duzy"])
    przechodzi_filtr = df.apply(przejdz_filtr_jakosci, axis=1)
    do_odrzucenia_jakosc = maska_ma_wlasny_filtr & ~przechodzi_filtr
    if do_odrzucenia_jakosc.sum() > 0:
        for typ in ["centrum_handlowe", "supermarket", "parking_duzy"]:
            n = (do_odrzucenia_jakosc & (df["source_layer"] == typ)).sum()
            if n > 0:
                print(f"  {typ}: odrzucono {n} nie spelniajacych progu jakosci/wielkosci")
    df = df[~do_odrzucenia_jakosc].copy()

    # ---------- etap 2: realny prog pojemnosci dla parkingow ----------
    df["capacity_liczba"] = pd.to_numeric(df["capacity"], errors="coerce")
    maska_parking = df["source_layer"] == "parking_duzy"
    do_usuniecia = maska_parking & df["capacity_liczba"].notna() & (df["capacity_liczba"] < PROG_POJEMNOSCI_PARKING)
    if do_usuniecia.sum() > 0:
        print(f"  Parkingi ponizej progu {PROG_POJEMNOSCI_PARKING} miejsc: usunieto {do_usuniecia.sum()}")
    df = df[~do_usuniecia].copy()

    # ---------- etap 2b: dostepnosc parkingu (access) ----------
    # Ladowarka przy parkingu ma sens tylko jesli parking jest realnie
    # PUBLICZNY. Brak tagu 'access' w OSM jest konwencjonalnie
    # interpretowany jako "brak ograniczenia" (domyslnie publiczny) -
    # odrzucamy tylko JAWNIE oznaczone jako 'private' lub 'no'.
    # UWAGA: maska_parking liczona TERAZ, na aktualnym (juz przefiltrowanym
    # etapem pojemnosci) df - poprzednia wersja uzywala starej maski sprzed
    # zmiany indeksu, co pandas sygnalizowal ostrzezeniem o reindeksacji.
    maska_parking = df["source_layer"] == "parking_duzy"
    if maska_parking.sum() > 0:
        rozklad_access = df.loc[maska_parking, "access"].fillna("(brak tagu - zakladamy publiczny)").value_counts()
        print(f"\n  Rozklad dostepnosci (access) dla parking_duzy:")
        for wartosc, n in rozklad_access.items():
            print(f"    {wartosc}: {n}")

    maska_prywatny = maska_parking & df["access"].isin(["private", "no"])
    if maska_prywatny.sum() > 0:
        print(f"  Parkingi jawnie oznaczone jako private/no: usunieto {maska_prywatny.sum()}")
    df = df[~maska_prywatny].copy()

    # ---------- etap 3: deduplikacja przestrzenna ----------
    wyniki = []
    for typ, grupa in df.groupby("source_layer"):
        prog_dla_typu = PROGI_DUPLIKATU_M.get(typ, PROG_DUPLIKATU_DOMYSLNY_M)
        przed = len(grupa)
        grupa_czysta = deduplikuj_typ(grupa.reset_index(drop=True), prog_dla_typu)
        po = len(grupa_czysta)
        if przed != po:
            print(f"  {typ}: usunieto {przed - po} duplikatow w promieniu {prog_dla_typu}m")
        wyniki.append(grupa_czysta)

    df_czysty = pd.concat(wyniki, ignore_index=True).drop(columns=["capacity_liczba"])
    print(f"Czyszczenie danych - po: {len(df_czysty)} lokalizacji")
    return df_czysty

# Migracja nazw source_layer - jesli kiedys zmienimy nazwe typu w
# ZAPYTANIA (tak jak supermarket_duzy -> supermarket), stary plik CSV
# nadal ma STARA nazwe. Bez tej migracji skrypt myslalby, ze nowej nazwy
# jeszcze nie ma, i pobieralby ja ponownie - dublujac dane pod dwiema
# etykietami (dokladnie to sie stalo przy poprzednim uruchomieniu).
PRZEMIANOWANIA_TYPOW = {
    "supermarket_duzy": "supermarket",
}

if __name__ == "__main__":
    # WZNAWIALNOSC: jesli plik juz istnieje (z poprzedniego, czesciowo
    # udanego uruchomienia), wczytujemy go i POMIJAMY typy, ktore juz
    # mamy - zeby nie tracic wczesniej zdobytych danych przy kazdym
    # ponownym uruchomieniu po bledzie serwera.
    try:
        df_istniejacy = pd.read_csv(PLIK_SUROWY)
        # migrujemy stare nazwy PRZED sprawdzeniem, co juz mamy
        liczba_przed_migracja = df_istniejacy["source_layer"].isin(PRZEMIANOWANIA_TYPOW).sum()
        if liczba_przed_migracja > 0:
            print(f"Migruje {liczba_przed_migracja} wierszy ze starymi nazwami typow: {PRZEMIANOWANIA_TYPOW}")
            df_istniejacy["source_layer"] = df_istniejacy["source_layer"].replace(PRZEMIANOWANIA_TYPOW)
        typy_juz_pobrane = set(df_istniejacy["source_layer"].unique())
        print(f"Znaleziono surowe dane (plik {PLIK_SUROWY}) z typami: {typy_juz_pobrane} - pomijam ponowne pobieranie.\n")
    except FileNotFoundError:
        df_istniejacy = pd.DataFrame()
        typy_juz_pobrane = set()

    wszystkie_wyniki = []
    for nazwa_typu, zapytanie in ZAPYTANIA.items():
        if nazwa_typu in typy_juz_pobrane:
            print(f"Pomijam {nazwa_typu} - juz obecne w pliku wynikowym.")
            continue
        try:
            wszystkie_wyniki.extend(pobierz_z_ponawianiem(nazwa_typu, zapytanie))
        except Exception as e:
            print(f"  BLAD dla {nazwa_typu} (po wyczerpaniu prob ponawiania): {e}")
        time.sleep(30)  # dluzsza, uprzejma przerwa miedzy TYPAMI zapytan

    df_nowe = pd.DataFrame(wszystkie_wyniki)
    df_surowy_polaczony = pd.concat([df_istniejacy, df_nowe], ignore_index=True)
    df_surowy_polaczony.to_csv(PLIK_SUROWY, index=False)
    print(f"Zapisano surowe dane: {PLIK_SUROWY} ({len(df_surowy_polaczony)} wierszy, bez czyszczenia)")

    # Czyszczenie ZAWSZE liczone od nowa z PELNYCH danych surowych - dzieki
    # temu kazda poprawka logiki oczysc_dane() od razu odda pelny efekt
    # na WSZYSTKICH danych, nie tylko na nowo pobranych w tym uruchomieniu.
    df_finalny = oczysc_dane(df_surowy_polaczony) if len(df_surowy_polaczony) > 0 else df_surowy_polaczony
    df_finalny.to_csv(PLIK_WYNIKOWY, index=False)
    print(f"\nZapisano {PLIK_WYNIKOWY}, {len(df_finalny)} lokalizacji lacznie")
    print(df_finalny["source_layer"].value_counts())

    brakujace = set(ZAPYTANIA.keys()) - set(df_finalny["source_layer"].unique() if len(df_finalny) else [])
    if brakujace:
        print(f"\nUWAGA: nadal brakuje typow: {brakujace} - uruchom skrypt ponownie, zeby je dobrac.")
    else:
        print("\nWszystkie typy pobrane pomyslnie.")

    # ---------- reczna weryfikacja: losowa probka z kazdego typu ----------
    # Kazdy typ ma inne pole "dowodowe" - to, ktore faktycznie decydowalo
    # o przejsciu filtru jakosci (patrz przejdz_filtr_jakosci) - zeby przy
    # rzucie oka bylo widac, CZY I DLACZEGO dana pozycja przeszla filtr,
    # nie tylko pojemnosc, ktora dla centrow/supermarketow zawsze jest pusta.
    # ---------- rozklad marek supermarketow - czy losowa probka Biedronki
    # to przypadek losowania, czy brak roznorodnosci w danych ----------
    if "supermarket" in df_finalny["source_layer"].unique():
        print(f"\n{'='*70}")
        print("PELNY ROZKLAD MAREK SUPERMARKETOW (nie tylko losowa probka)")
        print(f"{'='*70}")
        rozklad_marek = df_finalny[df_finalny["source_layer"] == "supermarket"]["brand"].value_counts()
        print(rozklad_marek.to_string())

    POLE_DOWODOWE = {
        "centrum_handlowe": "powierzchnia_m2",
        "supermarket": "brand",
        "parking_duzy": "capacity",
    }

    print(f"\n{'='*70}")
    print("LOSOWA PROBKA DO RECZNEJ WERYFIKACJI (po 5 z kazdego typu)")
    print(f"{'='*70}")
    for typ in sorted(df_finalny["source_layer"].unique()):
        podzbior = df_finalny[df_finalny["source_layer"] == typ]
        probka = podzbior.sample(min(5, len(podzbior)), random_state=42)
        print(f"\n--- {typ} (n={len(podzbior)}) ---")
        for _, wiersz in probka.iterrows():
            nazwa = wiersz["name"] if pd.notna(wiersz["name"]) else "(brak nazwy)"
            pole = POLE_DOWODOWE.get(typ, "capacity")
            dowod = f"{pole}={wiersz.get(pole, 'n/a')}"
            print(f"  {nazwa} | ({wiersz['latitude']:.4f}, {wiersz['longitude']:.4f}) | {dowod}")

Znaleziono surowe dane (plik ../data/nowe_typy_osm_surowe.csv) z typami: {'centrum_handlowe', 'supermarket', 'parking_duzy'} - pomijam ponowne pobieranie.

Pomijam centrum_handlowe - juz obecne w pliku wynikowym.
Pomijam supermarket - juz obecne w pliku wynikowym.
Pomijam parking_duzy - juz obecne w pliku wynikowym.
Zapisano surowe dane: ../data/nowe_typy_osm_surowe.csv (50247 wierszy, bez czyszczenia)

Czyszczenie danych - przed: 50247 lokalizacji
  centrum_handlowe: odrzucono 1212 nie spelniajacych progu jakosci/wielkosci
  supermarket: odrzucono 6092 nie spelniajacych progu jakosci/wielkosci
  Parkingi ponizej progu 50 miejsc: usunieto 31463

  Rozklad dostepnosci (access) dla parking_duzy:
    yes: 584
    (brak tagu - zakladamy publiczny): 466
    customers: 270
    private: 139
    permissive: 34
    permit: 18
    no: 5
    unknown: 2
    residents: 2
  Parkingi jawnie oznaczone jako private/no: usunieto 144
  centrum_handlowe: usunieto 20 duplikatow w promieniu 300m
  parking_d